# Athena++ Monte Carlo images

Bins photon lists into images and plots them, with the same code as `make_image.py` and
`plot_image.py`: the parameter dictionaries below mirror those scripts' options, the
notebook assembles the command line from them and calls the scripts' functions.  So
`python make_image.py -h` and `python plot_image.py -h` document every entry.

Two flags control what runs.  With `MAKE` on, the lists are read once and the images are
written to disk; with `MAKE` off and `PLOT` on, the images already on disk are plotted,
so the plotting cells can be re-run as often as needed without touching the lists.

Each photon is placed at its impact parameter in the image plane of a distant observer
along its direction, with the sky's axes (north up, east left), and binned by the cosine
of the observer's inclination and by photon energy.  The lists are given as files, any
number of them; the ranks of one output are summed, and with `combine` the outputs are
averaged weighted by their integration times.


In [ ]:
import glob
import os
import sys

import matplotlib.pyplot as plt
%matplotlib inline

# The Athena++ Monte Carlo python tools (athena_mc.py, mc_cli.py and the scripts).  Jupyter
# puts the notebook's own directory on the path, so nothing is needed while this notebook
# lives in vis/python/montecarlo; a copy kept next to a run sets the directory here.
ATHENA_VIS = None
if ATHENA_VIS is not None and ATHENA_VIS not in sys.path:
    sys.path.insert(0, ATHENA_VIS)

import athena_mc as athenamc
import mc_cli
import make_image
import plot_image


## What to do, and on which lists

In [ ]:
MAKE = True    # bin the photon lists into images
PLOT = True    # plot the images
SAVE = True    # also save each plot to a file

# The photon lists.  Ranks of one output are summed; with combine below, outputs are averaged.
LISTS = sorted(glob.glob('xrb.out1.proc*.list'))
print(f"{len(LISTS)} list file(s)")


## Making the images

The image plane runs from `-XMAX` to `XMAX` and `-YMAX` to `YMAX` in the units of the
list's positions, unless `xmin`/`ymin` are set.  The remaining entries are the options of
`make_image.py`; `None` and `False` leave an option out.


In [ ]:
XMAX, YMAX = 20.0, 20.0

make_params = dict(
    nx=64, ny=64,        # pixels across and down
    xmin=None, ymin=None,          # left and bottom edges; None gives -XMAX and -YMAX
    ninc=16, imin=-1.0, imax=1.0,  # bins in the cosine of the inclination
    nen=1, emin=1.0e-300, emax=1.0e300,   # logarithmic energy bins, keV
    unit=None,           # positions are code units; name a unit here to label the axes
    calclum=False,       # print the luminosity of each output from its lists
    combine=False,       # average all outputs into one image, weighted by their times
    outfile=None,        # output name; None gives <base>.<output>.img, or <base>.img combined
)


A screen leaves photons out.  It takes a `Photons` chunk and returns True for the photons
to drop.  `None` uses all photons.


In [ ]:
SCREEN = None


In [ ]:
IMAGES = None
if MAKE:
    make_args = make_image.parse_args(
        mc_cli.argv_from([XMAX, YMAX, LISTS], make_params))
    make_image.main(make_args, screen_function=SCREEN)
    IMAGES = make_args.outnames


## Plotting

One figure per image file, inclination bin and energy bin in the lists below.  `type` is
`intensity`, a Stokes plane `q`, `u` or `v`, `polfrac` or `polangle`; the Stokes
quantities are divided by the intensity unless `unnormalized`.  Everything else is as in
`plot_image.py`.  Saved plots are named after the image with the bin and type appended.


In [ ]:
PLOT_FILES = IMAGES if IMAGES is not None else sorted(glob.glob('xrb.out1*.img'))
IINC = [0]           # inclination bin(s) to plot
IE = [0]             # energy bin(s) to plot

plot_params = dict(
    type='intensity',    # intensity, q, u, v, polfrac, polangle
    unnormalized=False,  # show q, u, v, polfrac as stored rather than divided by I
    xmin=None, xmax=None, ymin=None, ymax=None,   # zoom; None is the image's extent
    colormap='hot',      # twilight suits the cyclic polarization angle
    vmin=None, vmax=None,          # color scale limits
    vnorm=False,         # divide by the maximum first
    logc=False,          # logarithmic color scale (polfrac and polangle then need vmin)
    pvec=False,          # draw polarization bars
    average=False,       # average the Stokes planes over each block of step pixels
    step=4,              # pixels between bars
)


In [ ]:
if PLOT:
    for file in PLOT_FILES:
        for iinc in IINC:
            for ie in IE:
                stem = os.path.splitext(file)[0]
                outfile = f"{stem}.i{iinc}.e{ie}.{plot_params['type']}.png"
                plot_args = plot_image.parse_args(mc_cli.argv_from(
                    [file], dict(plot_params, iinc=iinc, ie=ie, outfile=outfile)))
                fig = plot_image.make_figure(plot_args)
                fig.suptitle(f"{file}: cos i bin {iinc}, energy bin {ie}")
                if SAVE:
                    fig.savefig(outfile)
                    print(f"wrote {outfile}")
                plt.show()
